# ⛏️ 04 · 完整挖矿模拟器

> 把区块头 + Merkle 树 + PoW 组合在一起，亲手挖一个区块。

## 4.1 矿工模拟器

这个矿工做的事和真矿工一模一样：
1. 接收区块模板（区块头 + 交易列表）
2. 构造 coinbase 交易
3. 计算 merkle root
4. 暴力试 nonce
5. 找到后广播

In [ ]:
import hashlib, struct, time, random
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

# ---- Reuse earlier primitives ----
def sha256(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()

def double_sha256(data: bytes) -> bytes:
    return hashlib.sha256(hashlib.sha256(data).digest()).digest()

def bits_to_target(bits: int) -> int:
    return (bits & 0x00ffffff) * (256 ** ((bits >> 24) - 3))

def target_to_difficulty(target: int) -> float:
    return bits_to_target(0x1d00ffff) / target

@dataclass
class BlockHeader:
    version: int = 1
    prev_hash: bytes = field(default_factory=lambda: b"\x00" * 32)
    merkle_root: bytes = field(default_factory=lambda: b"\x00" * 32)
    timestamp: int = 0
    bits: int = 0x1d00ffff
    nonce: int = 0

    def serialize(self) -> bytes:
        return struct.pack("<I32s32sIII", self.version, self.prev_hash,
                          self.merkle_root, self.timestamp, self.bits, self.nonce)

    def hash(self) -> bytes:
        return double_sha256(self.serialize())[::-1]

def build_merkle_root(tx_data: List[bytes]) -> bytes:
    hashes = [double_sha256(d) for d in tx_data]
    while len(hashes) > 1:
        if len(hashes) % 2 == 1:
            hashes.append(hashes[-1])
        hashes = [double_sha256(hashes[i] + hashes[i+1])
                  for i in range(0, len(hashes), 2)]
    return hashes[0] if hashes else b"\x00" * 32

print("✅ Primitives ready")

In [ ]:
class Miner:
    def __init__(self, name: str, hashrate: int = 5_000_000):
        self.name = name
        self.hashrate = hashrate  # simulated H/s
        self.total_hashes = 0
        self.blocks_found = 0

    def mine(self, header: BlockHeader, max_seconds: float = 60.0) -> Optional[int]:
        target = bits_to_target(header.bits)
        start = time.time()
        last_report = start

        print(f"\n⛏️  {self.name} mining...")
        print(f"    Target: < 0x{target:064x}")
        print(f"    Difficulty: {target_to_difficulty(target):,.0f}")

        for nonce in range(2**32):
            header.nonce = nonce
            h_int = int.from_bytes(double_sha256(header.serialize()), "big")
            self.total_hashes += 1

            if h_int < target:
                elapsed = time.time() - start
                self.blocks_found += 1
                print(f"\n🎉 BLOCK FOUND!")
                print(f"    Nonce: {nonce:,}")
                print(f"    Hash:  {header.hash().hex()}")
                print(f"    Time:  {elapsed:.2f}s")
                print(f"    Hashrate: {self.total_hashes/elapsed:,.0f} H/s")
                return nonce

            now = time.time()
            if now - last_report >= 1:
                hps = self.total_hashes / (now - start)
                print(f"    {nonce:>12,} nonces | {self.total_hashes:>12,} hashes | "
                      f"{now-start:>6.1f}s | {hps:,.0f} H/s", end="\r")
                last_report = now

            if time.time() - start > max_seconds:
                print(f"\n⏰ Timeout after {max_seconds}s — {self.total_hashes:,} hashes tried")
                return None

        return None

print("✅ Miner class ready")

### 挖一个区块！

In [ ]:
# Prepare block template
txs_data = [
    f"Alice pays Bob {random.randint(1,50)} BTC".encode()
    for _ in range(5)
]
# Add coinbase
coinbase = f"COINBASE -> Miner_Reward {3.125} BTC".encode()
all_txs = [coinbase] + txs_data

header = BlockHeader(
    prev_hash=bytes.fromhex("0" * 64),
    merkle_root=build_merkle_root(all_txs),
    timestamp=int(time.time()),
    bits=0x2000ffff,  # easier than Bitcoin, takes seconds not centuries
)

miner = Miner("Your Miner", hashrate=10_000_000)
result = miner.mine(header, max_seconds=120)

if result is not None:
    print(f"\n{'='*50}")
    print("✅ SUCCESSFULLY MINED A BLOCK!")
    print(f"   Hash: {header.hash().hex()}")
    print(f"   Nonce: {result}")
    print(f"   Total hashes: {miner.total_hashes:,}")
    print(f"{'='*50}")

### 算力 vs 预期时间

相同难度，不同算力的预期挖矿时间差了数十亿倍。

In [ ]:
import matplotlib.pyplot as plt

def expected_mining_time(hashrate: float, target: int) -> float:
    return (2**256 / target) / hashrate

target = bits_to_target(0x2000ffff)

hashrates = {
    "CPU (Python)": 1_000_000,
    "CPU (C)": 10_000_000,
    "GPU (CUDA)": 3_000_000_000,
    "ASIC (1 chip)": 50_000_000_000_000,
    "ASIC (1 miner)": 200_000_000_000_000,
}

print("Hashrate source    | Expected time to mine")
print("-" * 45)
for name, hr in hashrates.items():
    t = expected_mining_time(hr, target)
    print(f"  {name:<18} | {t:.1f} seconds" if t < 60
          else f"  {name:<18} | {t/60:.1f} minutes" if t < 3600
          else f"  {name:<18} | {t/3600:.1f} hours" if t < 86400
          else f"  {name:<18} | {t/86400:.0f} days" if t < 86400*365
          else f"  {name:<18} | {t/86400/365:,.0f} YEARS")

## 小结

✅ 完整矿工只需 ~100 行 Python
✅ 核心循环：serialize → double SHA-256 → compare → nonce++
✅ 难度用 nBits 编码，目标值用 bits_to_target 解码
✅ Python 挖不了真比特币——但完全够理解原理

**下一课**: 矿池——一个人挖是买彩票，一群人挖是开公司。